DS4002 Image analysis

use master_release_schedule_update4 for baseline, filter for films with valid box office info

In [ ]:
import pandas as pd

# 1. Load your dataset
filename = 'MASTER_RELEASE_SCHEDULE_UPDATE4.csv'
df = pd.read_csv(filename)

# 2. Filter the dataset
# We keep rows where at least one of these three columns is NOT null (notna)
# .any(axis=1) returns True if at least one column in that row is True
financial_columns = ['domestic', 'worldwide', 'international']
filtered_df = df[df[financial_columns].notna().any(axis=1)]

# 3. Report the results
original_count = len(df)
filtered_count = len(filtered_df)
removed_count = original_count - filtered_count

print(f"Original entries: {original_count}")
print(f"Entries kept (with financial data): {filtered_count}")
print(f"Entries removed (no financial data): {removed_count}")

# 4. Save the reduced dataset to a new CSV
filtered_df.to_csv('FILTERED_RELEASE_SCHEDULE.csv', index=False)
print("Saved reduced dataset to 'FILTERED_RELEASE_SCHEDULE.csv'")

Use this dataset to get a random sample of 1000 film posters

In [ ]:
import pandas as pd
import requests
import os
import time
from tqdm import tqdm

# --- Configuration ---
API_KEY = "40b746fe3d756e4c2d54323e530a5911"
INPUT_FILE = "FILTERED_RELEASE_SCHEDULE.csv"
SAVE_DIR = "sampled_posters"
BASE_IMAGE_URL = "https://image.tmdb.org/t/p/w500"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 1. Load and Filter Dataset
df = pd.read_csv(INPUT_FILE)

# Keep rows where at least one financial column is not null
financial_cols = ['domestic', 'worldwide', 'international']
df_filtered = df[df[financial_cols].notna().any(axis=1)].copy()

# 2. Take a Random Sample of 100
# If the dataset has fewer than 100 rows, it will take all of them
sample_size = min(1000, len(df_filtered))
df_sample = df_filtered.sample(n=sample_size, random_state=42)

print(f"Filtered dataset from {len(df)} to {len(df_filtered)} films with financial data.")
print(f"Proceeding with a random sample of {sample_size} films.")

def get_poster_by_imdb_id(tconst):
    """Uses TMDb Find endpoint to get movie poster via IMDb ID."""
    # The 'find' endpoint is perfect for tconst lookups
    url = f"https://api.themoviedb.org/3/find/{tconst}"
    params = {
        "api_key": API_KEY,
        "external_source": "imdb_id"
    }
    
    try:
        response = requests.get(url, params=params).json()
        # Find results are grouped by type (movie_results, tv_results, etc.)
        movie_results = response.get('movie_results', [])
        
        if movie_results:
            path = movie_results[0].get('poster_path')
            if path:
                return f"{BASE_IMAGE_URL}{path}"
    except Exception as e:
        print(f"Error finding ID {tconst}: {e}")
    return None

# 3. Download Loop
results = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    tconst = row['tconst']
    title = row['primaryTitle']
    
    poster_url = get_poster_by_imdb_id(tconst)
    
    if poster_url:
        # Save filename using tconst to ensure it's unique and valid
        filename = f"{tconst}.jpg"
        save_path = os.path.join(SAVE_DIR, filename)
        
        try:
            img_data = requests.get(poster_url).content
            with open(save_path, 'wb') as f:
                f.write(img_data)
            
            # Record success
            row_data = row.to_dict()
            row_data['local_poster_path'] = save_path
            results.append(row_data)
        except Exception as e:
            print(f"Failed to download {title}: {e}")
    
    # Respect rate limits
    time.sleep(0.1)

# 4. Save the Final Mapping
final_df = pd.DataFrame(results)
final_df.to_csv("SAMPLED_POSTER_METADATA.csv", index=False)
print(f"\nSuccess! {len(final_df)} posters downloaded.")

This code chunk implements an unsupervised machine learning pipeline to quantify the visual aesthetic of movie posters. By converting subjective "art" into objective "data," it allows for statistical testing of color-based hypotheses.

Core Logic:

Image Preprocessing: Uses OpenCV to load posters and resize them to a standard width (200px). This maintains color integrity while significantly reducing the computational load for the clustering algorithm.

K-Means Clustering: Employs the scikit-learn K-Means algorithm to group millions of pixels into k=5 dominant color clusters. This effectively "summarizes" the poster's primary color scheme.

Feature Engineering: Extracts the RGB coordinates for each cluster center and sorts them by frequency, providing a weighted profile of the poster's most prominent hues.

Data Integration: Maps these color palettes back to the original film metadata (indexed by tconst), producing a final dataset (FINAL_ANALYSIS_READY_DATA.csv) ready for correlation analysis with box office revenue and release schedules.

In [ ]:
import cv2
import pandas as pd
import numpy as np
import os
from sklearn.cluster import KMeans
from tqdm import tqdm

# --- Configuration ---
IMAGE_DIR = "sampled_posters"
METADATA_FILE = "SAMPLED_POSTER_METADATA.csv"
CLUSTERS = 5 # Number of dominant colors to find per poster

# Load the metadata we created in the last step
df = pd.read_csv(METADATA_FILE)

def get_dominant_colors(image_path, k=5):
    """Loads an image and returns the k-dominant RGB colors."""
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    # 1. Convert BGR (OpenCV default) to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 2. Resize to speed up processing (200px width is enough for color analysis)
    height, width, _ = img.shape
    new_width = 200
    new_height = int(height * (new_width / width))
    img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_AREA)
    
    # 3. Reshape the image to be a list of pixels
    pixels = img.reshape((-1, 3))
    
    # 4. Cluster pixels using KMeans
    kmeans = KMeans(n_init=10, n_clusters=k, random_state=42)
    kmeans.fit(pixels)
    
    # Get colors (cluster centers) and convert to integer
    colors = kmeans.cluster_centers_.astype(int)
    
    # Sort colors by how frequently they appear (optional but helpful)
    labels = list(kmeans.labels_)
    counts = [labels.count(i) for i in range(k)]
    sorted_colors = [colors[i] for i in np.argsort(counts)[::-1]]
    
    return sorted_colors

# --- Processing Loop ---
print(f"Analyzing color palettes for {len(df)} posters...")
palette_data = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    path = row['local_poster_path']
    
    if os.path.exists(path):
        palette = get_dominant_colors(path, k=CLUSTERS)
        # Store as a list of RGB tuples
        palette_data.append(palette)
    else:
        palette_data.append(None)

# Add the new column and save
df['color_palette_rgb'] = palette_data
df.to_csv("FINAL_ANALYSIS_READY_DATA.csv", index=False)
print("Analysis complete! Palettes saved to 'FINAL_ANALYSIS_READY_DATA.csv'")

This stage of the pipeline quantifies the "human element" of movie posters to test the correlation between star-driven marketing and financial returns.

Logic: The script utilizes the Haar Cascade object detection algorithm via OpenCV. By converting posters to grayscale and applying a multi-scale sliding window, the model identifies rectangular regions that match the visual patterns of human faces.

Data Transformation: Subjective visual density is converted into a discrete numerical variable (face_count).

Merging: This new variable is joined with the existing TMDb metadata and K-Means color clusters, creating a rich, multimodal dataset.

Research Application: This allows for testing whether "ensemble" posters (high face count) correlate with higher Worldwide Revenue compared to minimalist or landscape-focused posters.

In [ ]:
import cv2
import pandas as pd
import os
from tqdm import tqdm

# --- Configuration ---
INPUT_FILE = "FINAL_ANALYSIS_READY_DATA.csv"
OUTPUT_FILE = "COMPLETE_ANALYSIS_DATA.csv"

# Load the XML file for face detection (standard OpenCV feature)
# Note: Ensure 'haarcascade_frontalface_default.xml' is available or use the cv2 path
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# 1. Load your existing analysis-ready data
df = pd.read_csv(INPUT_FILE)

def count_faces(image_path):
    """Detects and counts frontal faces in a movie poster."""
    img = cv2.imread(image_path)
    if img is None:
        return 0
    
    # Convert to grayscale for the face detector
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Detect faces
    # scaleFactor: How much the image size is reduced at each image scale
    # minNeighbors: How many neighbors each candidate rectangle should have to retain it
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    
    return len(faces)

# --- Processing Loop ---
print(f"Detecting faces for {len(df)} posters...")
face_counts = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    path = row['local_poster_path']
    
    if os.path.exists(path):
        count = count_faces(path)
        face_counts.append(count)
    else:
        face_counts.append(0)

# 2. Merge and Save
df['face_count'] = face_counts
df.to_csv(OUTPUT_FILE, index=False)

print(f"Success! Merged data saved to '{OUTPUT_FILE}'.")
print(df[['primaryTitle', 'face_count', 'worldwide']].head())

The above face detection does not seem to work

To address the accuracy limitations of traditional edge-based detection (Haar Cascades), the pipeline was upgraded to use MediaPipe Face Detection, a specialized Short-range BlazeFace model.

Logic: Unlike older methods that rely on simple contrast patterns, this system uses a Single Shot Detector (SSD) architecture optimized for mobile and web real-time performance. This allows it to identify faces with higher precision even when they are partially obscured, rotated, or stylized.

Performance: The model_selection=1 parameter was utilized to optimize for "full-range" detection, ensuring that small background characters and central "floating heads" are both captured in the face_count metric.

Data Integration: The refined counts were merged into the master dataframe, creating a high-fidelity feature set for the final correlation analysis between visual density and box office returns.

In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import os
from tqdm import tqdm
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# --- Configuration ---
INPUT_FILE  = "FINAL_ANALYSIS_READY_DATA.csv"
OUTPUT_FILE = "COMPLETE_ANALYSIS_DATA_V2.csv"

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Could not find {INPUT_FILE}.")

# Download the model file if needed (one-time download ~6 MB)
MODEL_PATH = "blaze_face_short_range.tflite"
if not os.path.exists(MODEL_PATH):
    import urllib.request
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
    print(f"Downloading model to {MODEL_PATH}...")
    urllib.request.urlretrieve(url, MODEL_PATH)

# Build the detector using the new Tasks API
base_options    = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
detector_options = vision.FaceDetectorOptions(
    base_options=base_options,
    min_detection_confidence=0.5
)
face_detector = vision.FaceDetector.create_from_options(detector_options)

df = pd.read_csv(INPUT_FILE)

def count_faces_mediapipe(image_path):
    """Detects faces using the MediaPipe Tasks FaceDetector."""
    img = cv2.imread(image_path)
    if img is None:
        return 0
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Wrap in MediaPipe Image format
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    results  = face_detector.detect(mp_image)
    return len(results.detections) if results.detections else 0

# --- Processing Loop (unchanged) ---
print(f"Running face detection for {len(df)} posters...")
face_counts = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    path = row["local_poster_path"]
    face_counts.append(
        count_faces_mediapipe(path)
        if isinstance(path, str) and os.path.exists(path)
        else 0
    )

df["face_count_refined"] = face_counts
df.to_csv(OUTPUT_FILE, index=False)
face_detector.close()
print(f"Success! Saved to '{OUTPUT_FILE}'.")


above face detection system also is not accurate

## RetinaFace: Deep Learning Face Detection for Movie Posters

To overcome the limitations of both Haar Cascades and MediaPipe BlazeFace, the pipeline was upgraded to **RetinaFace**, a state-of-the-art single-stage face detector.

**Why RetinaFace?**
- Uses a **ResNet-50 backbone** trained on the WIDER FACE benchmark, giving it far greater robustness than lightweight mobile models.
- Jointly detects faces *and* five facial landmarks (eyes, nose, mouth corners), making it reliable for partially occluded, profile, or stylized faces — all common in artistic movie posters.
- Significantly outperforms Haar Cascades and BlazeFace on challenging, real-world imagery.

**Output:** The `face_count_retinaface` column records the number of faces detected per poster. Results are merged with the existing metadata and saved to `COMPLETE_ANALYSIS_DATA_V3.csv`.

In [ ]:
import os

# IMPORTANT: Must be set before any TensorFlow/Keras import.
# Forces TF 2.16+ to use legacy Keras 2 (tf-keras) instead of Keras 3,
# which is required for the retinaface package.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Set working directory to where the data and posters live
os.chdir("/Users/rowanrosenblum/PycharmProjects/JupyterProject")
print(f"Working directory: {os.getcwd()}")

from retinaface import RetinaFace
import pandas as pd
from tqdm import tqdm

# --- Configuration ---
INPUT_FILE  = "FINAL_ANALYSIS_READY_DATA.csv"
OUTPUT_FILE = "COMPLETE_ANALYSIS_DATA_V3.csv"

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Could not find {INPUT_FILE}. Make sure the K-Means color step has been run first.")

df = pd.read_csv(INPUT_FILE)

def count_faces_retinaface(image_path):
    """
    Detects faces in a movie poster using RetinaFace.

    RetinaFace.detect_faces() returns:
      - A dict keyed by face index (e.g. {'face_1': {...}, 'face_2': {...}}) when faces are found
      - False or an empty result when no faces are detected
    """
    try:
        faces = RetinaFace.detect_faces(image_path)
        if isinstance(faces, dict):
            return len(faces)
        return 0
    except Exception as e:
        print(f"Error on {image_path}: {e}")
        return 0

# --- Processing Loop ---
# Note: RetinaFace uses a ResNet-50 backbone and runs on CPU by default.
# Expect ~1-3 seconds per image (~15-30 min for 1000 posters without a GPU).
print(f"Running RetinaFace detection for {len(df)} posters...")

face_counts = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    path = row["local_poster_path"]
    face_counts.append(
        count_faces_retinaface(path)
        if isinstance(path, str) and os.path.exists(path)
        else 0
    )

# --- Save Results ---
df["face_count_retinaface"] = face_counts
df.to_csv(OUTPUT_FILE, index=False)

print(f"\nSuccess! Saved to '{OUTPUT_FILE}'.")
print(f"Faces detected — median: {pd.Series(face_counts).median()}, max: {pd.Series(face_counts).max()}")
print(df[["primaryTitle", "face_count_retinaface", "worldwide"]].head(10).to_string(index=False))


## Feature Engineering

This step derives all analysis-ready variables from the raw image data and existing columns in `COMPLETE_ANALYSIS_DATA_V3.csv`.

**From `color_palette_rgb`:**
- `dom_r`, `dom_g`, `dom_b` — RGB components of the single most dominant color
- `avg_brightness` — mean brightness (HSV V channel) across all pixels
- `avg_saturation` — mean saturation (HSV S channel) across all pixels
- `orange_teal_score` — proportion of pixels whose hue falls in orange (5–25°) or teal (170–200°) ranges

**From `face_count_retinaface` and image size:**
- `face_density` — face count divided by image area (faces per 10,000 pixels)

**From `worldwide` / `domestic`:**
- `log_worldwide`, `log_domestic` — log1p-transformed revenue to reduce skewness

**From `first_us_theatrical_date`:**
- `release_window` — categorical: Summer (May–Aug), Awards (Sep–Nov), Holiday (Dec), Spring (Mar–Apr), Other

**Output:** `FEATURE_ENGINEERED_DATA.csv`

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

os.chdir("/Users/rowanrosenblum/PycharmProjects/JupyterProject")

INPUT_FILE  = "COMPLETE_ANALYSIS_DATA_V3.csv"
OUTPUT_FILE = "FEATURE_ENGINEERED_DATA.csv"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows from {INPUT_FILE}")

# ── 1. Parse dominant color from color_palette_rgb ──────────────────────────
def parse_dominant_color(palette_str):
    """Returns the RGB of the most dominant (first) cluster center."""
    try:
        palette = ast.literal_eval(palette_str)
        if palette and len(palette) > 0:
            c = palette[0]  # already sorted by frequency
            return int(c[0]), int(c[1]), int(c[2])
    except Exception:
        pass
    return None, None, None

dom = df["color_palette_rgb"].apply(parse_dominant_color)
df["dom_r"] = [x[0] for x in dom]
df["dom_g"] = [x[1] for x in dom]
df["dom_b"] = [x[2] for x in dom]

# ── 2. Per-image: brightness, saturation, orange-teal score ─────────────────
def extract_image_features(image_path):
    """
    Returns (avg_brightness, avg_saturation, orange_teal_score).
    Brightness and saturation are means of the V and S channels (0-255).
    Orange-teal score is the proportion of pixels whose hue is in
    orange (5-25 deg) or teal (170-200 deg) ranges in OpenCV hue space
    (which is 0-180, so divide degrees by 2).
    """
    img = cv2.imread(image_path)
    if img is None:
        return None, None, None

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).reshape(-1, 3)
    h, s, v = hsv[:, 0], hsv[:, 1], hsv[:, 2]

    avg_brightness  = float(np.mean(v))
    avg_saturation  = float(np.mean(s))

    # OpenCV hue is 0-180 (divide degrees by 2)
    orange_mask = (h >= 3)  & (h <= 13)   # ~5-25 deg
    teal_mask   = (h >= 85) & (h <= 100)  # ~170-200 deg
    orange_teal_score = float(np.mean(orange_mask | teal_mask))

    return avg_brightness, avg_saturation, orange_teal_score

print("Extracting per-image color features...")
brightness_list, saturation_list, ot_score_list = [], [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    path = row["local_poster_path"]
    if isinstance(path, str) and os.path.exists(path):
        b, s, ot = extract_image_features(path)
    else:
        b, s, ot = None, None, None
    brightness_list.append(b)
    saturation_list.append(s)
    ot_score_list.append(ot)

df["avg_brightness"]   = brightness_list
df["avg_saturation"]   = saturation_list
df["orange_teal_score"] = ot_score_list

# ── 3. Face density (faces per 10,000 pixels) ───────────────────────────────
def get_image_area(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    h, w, _ = img.shape
    return h * w

print("Computing face density...")
areas = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    path = row["local_poster_path"]
    areas.append(get_image_area(path) if isinstance(path, str) and os.path.exists(path) else None)

df["image_area"] = areas
df["face_density"] = df["face_count_retinaface"] / (df["image_area"] / 10_000)

# ── 4. Log-transformed revenue ───────────────────────────────────────────────
df["log_worldwide"] = np.log1p(pd.to_numeric(df["worldwide"], errors="coerce"))
df["log_domestic"]  = np.log1p(pd.to_numeric(df["domestic"],  errors="coerce"))

# ── 5. Release window ────────────────────────────────────────────────────────
def classify_release_window(date_str):
    try:
        month = pd.to_datetime(date_str).month
        if month in [5, 6, 7, 8]:   return "Summer"
        if month in [9, 10, 11]:    return "Awards"
        if month == 12:             return "Holiday"
        if month in [3, 4]:         return "Spring"
        return "Other"
    except Exception:
        return None

df["release_window"] = df["first_us_theatrical_date"].apply(classify_release_window)

# ── 6. Save ──────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved to '{OUTPUT_FILE}' with {len(df)} rows.")

# Summary of new features
new_cols = ["dom_r", "dom_g", "dom_b", "avg_brightness", "avg_saturation",
            "orange_teal_score", "face_density", "log_worldwide", "log_domestic", "release_window"]
print(df[new_cols].describe(include="all").round(3))
